# ECG Deployment Notebook

Takes the artifacts produced by `training.ipynb` and the chosen winning model
from `comparison.ipynb`, and ships everything to HuggingFace: the model weights
to Model Hub repo, and the Gradio demo to a spaces repo.

Pipeline:
Setup + HF login

Pick which model to deploy (usually the ensemble-best single model)

Export example ECGs as CSV (for the Space's example gallery)

Push model + normalisation params + thresholds to the Model Hub

Write Space files (`app.py`, `requirements.txt`, `README.md`)

Push Space files to the Space repo

Load everything back from HF Hub and run one prediction

Prerequisites:
- `training.ipynb` has been run (so `artifacts/` exists)
- `comparison.ipynb` has been run (so we know which model won + thresholds)
- You have an HF account and a write token from https://huggingface.co/settings/tokens

## 1. Setup

In [1]:
# Install HF tooling if needed
# !pip install -q huggingface_hub gradio

In [2]:
import os, json, shutil, pickle
import numpy as np
import pandas as pd
import keras
from huggingface_hub import hf_hub_download
from sklearn.metrics import f1_score
from huggingface_hub import create_repo, upload_folder, login
os.environ["KERAS_BACKEND"] = "tensorflow"

# Paths
ART_DIR         = "artifacts"
SPACE_DIR       = "hf_space"       # local staging for the Space repo
MODEL_DIR       = "hf_model"       # local staging for the Model Hub repo
EXAMPLES_DIR    = os.path.join(SPACE_DIR, "examples")

# HF repo names
HF_USERNAME     = "Steenslid"
HF_MODEL_REPO   = f"{HF_USERNAME}/ecg-ptbxl-classification"
HF_SPACE_REPO   = f"{HF_USERNAME}/ecg-ptbxl-demo"

SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF",
              "V1", "V2", "V3", "V4", "V5", "V6"]

os.makedirs(SPACE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(EXAMPLES_DIR, exist_ok=True)

# Log in | will prompt for token if not already set
login()

## Choose which model to deploy

Run `comparison.ipynb` first to see which variant performs best. Set
`DEPLOY_MODEL` below to one of:

- `"cnn_aug"`    | 1D-CNN trained with augmentation
- `"cnn_noaug"`  | 1D-CNN trained without augmentation
- `"lstm_aug"`   | BiLSTM trained with augmentation
- `"lstm_noaug"` | BiLSTM trained without augmentation


In [3]:
# DEPLOY_MODEL format: "<arch>_<variant>" where arch in {cnn, lstm} and variant in {aug, noaug}
DEPLOY_MODEL = "cnn_noaug"

src_model_path = os.path.join(ART_DIR, f"ecg_{DEPLOY_MODEL}_final.keras")
src_norm_path  = os.path.join(ART_DIR, "normalisation_params.npz")
assert os.path.exists(src_model_path), f"Model file missing: {src_model_path}"
assert os.path.exists(src_norm_path),  f"Normalisation file missing: {src_norm_path}"

print(f"Deploying model: {DEPLOY_MODEL}")
print(f"  weights: {src_model_path}  ({os.path.getsize(src_model_path)/1e6:.1f} MB)")
print(f"  norm   : {src_norm_path}")


Deploying model: cnn_noaug
  weights: artifacts/ecg_cnn_noaug_final.keras  (8.4 MB)
  norm   : artifacts/normalisation_params.npz


## Export per-class thresholds

The comparison notebook computed per-class optimal thresholds. We save them
as a JSON file so `app.py` can load them and make predictions with the right
cutoffs

In [4]:
model = keras.saving.load_model(src_model_path, compile=False)
data = np.load(os.path.join(ART_DIR, "eval_sets.npz"))
X_val, y_val = data["X_val"], data["y_val"]

probs_val = model.predict(X_val, batch_size=128, verbose=0)

def find_optimal_thresholds(y_true, y_prob, grid=np.arange(0.05, 0.95, 0.01)):
    thresholds = np.zeros(y_true.shape[1])
    for i in range(y_true.shape[1]):
        f1s = [f1_score(y_true[:, i], (y_prob[:, i] >= t).astype(float),
                        zero_division=0) for t in grid]
        thresholds[i] = grid[int(np.argmax(f1s))]
    return thresholds

optimal_thresholds = find_optimal_thresholds(y_val, probs_val)
thresholds_dict = dict(zip(SUPERCLASSES, [float(t) for t in optimal_thresholds]))

with open(os.path.join(ART_DIR, "thresholds.json"), "w") as f:
    json.dump(thresholds_dict, f, indent=2)

print("Per-class thresholds (selected on validation):")
for sc, t in thresholds_dict.items():
    print(f"  {sc:5s}: {t:.3f}")

I0000 00:00:1776522083.115134   46973 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776522083.202495   46973 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776522084.684996   46973 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776522085.842731   46973 gpu_device.cc:2043] Created device /job:localhost/replica:0/task

Per-class thresholds (selected on validation):
  NORM : 0.270
  MI   : 0.350
  STTC : 0.450
  CD   : 0.440
  HYP  : 0.270


## Export example ECGs as CSV

Grab one cleanly-labelled example per class (NORM, MI, STTC) from the test
set to bundle with the Space. The app loads these as the "try a preloaded
example" gallery.

In [5]:
# Load raw (un-normalised) signals values when plotted. The app re-applies normalisation before inference.
# We reconstruct raw test signals by inverting the z-score on X_test.
X_test = data["X_test"]
y_test = data["y_test"]

norm = np.load(src_norm_path)
MEAN = norm["mean"]   # (1, 1, 12)
STD  = norm["std"]    # (1, 1, 12)
X_test_raw = X_test * STD + MEAN   # back to mV

def find_example(cls_idx, n_other_max=0):
    """First test sample with class_idx positive and ≤ n_other_max other labels."""
    for i in range(len(y_test)):
        y = y_test[i]
        if y[cls_idx] == 1 and (y.sum() - 1) <= n_other_max:
            return i
    return None

targets = [
    ("norm_example.csv", "NORM"),
    ("mi_example.csv",   "MI"),
    ("sttc_example.csv", "STTC"),
]

for fname, sc in targets:
    idx = find_example(SUPERCLASSES.index(sc))
    if idx is None:
        print(f"No clean {sc} sample found; skipping.")
        continue
    signal = X_test_raw[idx]   # (1000, 12), raw mV
    df_ecg = pd.DataFrame(signal, columns=LEAD_NAMES)
    out_path = os.path.join(EXAMPLES_DIR, fname)
    df_ecg.to_csv(out_path, index=False)
    active = [SUPERCLASSES[j] for j in range(5) if y_test[idx, j] == 1]
    print(f"  {fname}  (test idx {idx}, labels: {active})")

  norm_example.csv  (test idx 0, labels: ['NORM'])
  mi_example.csv  (test idx 5, labels: ['MI'])
  sttc_example.csv  (test idx 14, labels: ['STTC'])


## Push model artifacts to the HuggingFace Hub

We push one model (the chosen one) + normalisation params + thresholds
to the Model Hub. The Space will download these at startup.

In [6]:
MODEL_FILENAME = "ecg_model.keras"

shutil.copy(src_model_path, os.path.join(MODEL_DIR, MODEL_FILENAME))
shutil.copy(src_norm_path,  os.path.join(MODEL_DIR, "normalisation_params.npz"))
shutil.copy(os.path.join(ART_DIR, "thresholds.json"),
            os.path.join(MODEL_DIR, "thresholds.json"))

# Architecture name for the model card (strip _aug / _noaug suffix)
arch = DEPLOY_MODEL.rsplit("_", 1)[0].upper()  # "LSTM" or "CNN"
variant = DEPLOY_MODEL.rsplit("_", 1)[1]       # "aug" or "noaug"

# ── Write model card ──
model_card = f"""---
tags:
  - ecg
  - cardiovascular
  - multi-label-classification
  - keras
  - ptb-xl
datasets:
  - ptb-xl
license: mit
---

# ECG Cardiovascular Disease Classification

Multi-label classification of 5 cardiovascular superclasses
(NORM, MI, STTC, CD, HYP) from 12-lead ECG recordings, trained on PTB-XL.

**Deployed model**: {arch} ({variant} training variant)

## Files

- `{MODEL_FILENAME}` | trained  model
- `normalisation_params.npz` | per-channel mean and std (z-score, from training fold)
- `thresholds.json` | per-class decision thresholds optimised on the validation fold

## Usage

```python
import keras, numpy as np, json
from huggingface_hub import hf_hub_download

model = keras.saving.load_model(
    hf_hub_download("{HF_MODEL_REPO}", "{MODEL_FILENAME}"))
params = np.load(hf_hub_download("{HF_MODEL_REPO}", "normalisation_params.npz"))
with open(hf_hub_download("{HF_MODEL_REPO}", "thresholds.json")) as f:
    thresholds = json.load(f)

# Input x: (1000, 12) float32 ECG in mV, 100 Hz, standard 12-lead order
x_norm = (x - params["mean"]) / params["std"]
probs = model.predict(x_norm[np.newaxis])[0]
preds = {{sc: probs[i] >= thresholds[sc] for i, sc in enumerate(
    ["NORM","MI","STTC","CD","HYP"])}}
```

**Authors:** Edvard Vindenes Steenslid & Morten Kvamme
"""

with open(os.path.join(MODEL_DIR, "README.md"), "w") as f:
    f.write(model_card)

print("Model staging dir contents:")
for fn in sorted(os.listdir(MODEL_DIR)):
    size = os.path.getsize(os.path.join(MODEL_DIR, fn)) / 1e6
    print(f"  {fn:35s} {size:>8.2f} MB")


Model staging dir contents:
  README.md                               0.00 MB
  ecg_model.keras                         8.43 MB
  normalisation_params.npz                0.00 MB
  thresholds.json                         0.00 MB


In [7]:
from huggingface_hub import HfApi

api = HfApi()
create_repo(HF_MODEL_REPO, repo_type="model", exist_ok=True)

# Known files that may linger from older deployments.
STALE_MODEL_FILES = [
    "ecg_cnn_final.keras",
    "ecg_lstm_final.keras",
    "ecg_rescnn_final.keras",
    "ecg_xresnet1d_final.keras",
    "ecg_cnn_aug_final.keras",
    "ecg_cnn_noaug_final.keras",
    "ecg_lstm_aug_final.keras",
    "ecg_lstm_noaug_final.keras",
]

try:
    existing = set(api.list_repo_files(repo_id=HF_MODEL_REPO, repo_type="model"))
except Exception as e:
    print(f"(Repo is new or unreachable; skipping cleanup: {e})")
    existing = set()

to_delete = [f for f in STALE_MODEL_FILES if f in existing]
if to_delete:
    print(f"Deleting {len(to_delete)} stale file(s) from {HF_MODEL_REPO}:")
    for f in to_delete:
        print(f"  - {f}")
        api.delete_file(path_in_repo=f, repo_id=HF_MODEL_REPO,
                        repo_type="model",
                        commit_message=f"Remove stale {f}")
else:
    print("No stale files to delete in model repo.")

# Push fresh artifacts
upload_folder(
    folder_path=MODEL_DIR,
    repo_id=HF_MODEL_REPO,
    repo_type="model",
    commit_message=f"Deploy {DEPLOY_MODEL} with val-tuned thresholds",
)
print(f"\nPushed to https://huggingface.co/{HF_MODEL_REPO}")


Deleting 2 stale file(s) from Steenslid/ecg-ptbxl-classification:
  - ecg_cnn_final.keras
  - ecg_lstm_final.keras


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Pushed to https://huggingface.co/Steenslid/ecg-ptbxl-classification


## Write Space files (`app.py`, `requirements.txt`, `README.md`)

The Space pulls the model from the Model Hub at startup

In [8]:
app_py = '''"""ECG Classification Gradio App — HuggingFace Spaces
Deploys on HF Spaces (SDK: gradio). Run locally with: python app.py
"""
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import json
import keras
import numpy as np
import pandas as pd
import gradio as gr
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

# ── Constants ─────────────────────────────────────────────────────────
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
SUPERCLASS_NAMES = {
    "NORM": "Normal ECG",
    "MI":   "Myocardial Infarction",
    "STTC": "ST/T Change",
    "CD":   "Conduction Disturbance",
    "HYP":  "Hypertrophy",
}
LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF",
              "V1", "V2", "V3", "V4", "V5", "V6"]
SAMPLING_RATE = 100
HF_MODEL_REPO = "''' + HF_MODEL_REPO + '''"

# ── Load model + normalisation + thresholds from HF Hub ───────────────
print("Downloading artifacts from HF Hub...")
model_path = hf_hub_download(repo_id=HF_MODEL_REPO, filename="ecg_model.keras")
norm_path  = hf_hub_download(repo_id=HF_MODEL_REPO, filename="normalisation_params.npz")
thr_path   = hf_hub_download(repo_id=HF_MODEL_REPO, filename="thresholds.json")

model = keras.saving.load_model(model_path, compile=False)
params = np.load(norm_path)
MEAN = params["mean"].squeeze()
STD  = params["std"].squeeze()
with open(thr_path) as f:
    THRESHOLDS = json.load(f)
print("Loaded. Thresholds:", THRESHOLDS)


def load_ecg_from_csv(path):
    """Load CSV into (1000, 12) float32 array. Handles header/index variants."""
    try:
        df = pd.read_csv(path)
        if not np.issubdtype(df.dtypes.iloc[0], np.number):
            df = df.iloc[:, 1:]
    except Exception:
        df = pd.read_csv(path, header=None)
    arr = df.to_numpy(dtype=np.float32)
    if arr.shape[1] < 12:
        raise ValueError(f"Need 12 columns (leads), got {arr.shape[1]}.")
    arr = arr[:, :12]
    if arr.shape[0] != 1000:
        raise ValueError(f"Need 1000 rows (10s @ 100 Hz), got {arr.shape[0]}.")
    return arr


def plot_ecg(ecg):
    fig, axes = plt.subplots(4, 3, figsize=(12, 8), sharex=True)
    t = np.arange(ecg.shape[0]) / SAMPLING_RATE
    for i, ax in enumerate(axes.flat):
        ax.plot(t, ecg[:, i], linewidth=0.7, color="#2c3e50")
        ax.set_title(f"Lead {LEAD_NAMES[i]}", fontsize=9)
        ax.grid(alpha=0.25)
        if i >= 9: ax.set_xlabel("Time (s)")
        if i % 3 == 0: ax.set_ylabel("mV")
    fig.tight_layout()
    return fig


def predict_ecg(file):
    if file is None:
        return {"error": "No file uploaded."}, None
    try:
        ecg = load_ecg_from_csv(file.name)
    except Exception as e:
        return {"error": str(e)}, None

    ecg_norm = (ecg - MEAN) / STD
    probs = model.predict(ecg_norm[np.newaxis], verbose=0).squeeze()

    # Apply per-class thresholds for the diagnosis string
    diag = [sc for i, sc in enumerate(SUPERCLASSES) if probs[i] >= THRESHOLDS[sc]]
    diag_str = ", ".join(diag) if diag else "No class above threshold"

    result = {
        f"{sc} — {SUPERCLASS_NAMES[sc]} (thr={THRESHOLDS[sc]:.2f})": float(probs[i])
        for i, sc in enumerate(SUPERCLASSES)
    }
    fig = plot_ecg(ecg)
    return result, fig, f"**Predicted labels:** {diag_str}"


# ── UI ────────────────────────────────────────────────────────────────
EXAMPLES_DIR = "examples"
example_files = (sorted(os.path.join(EXAMPLES_DIR, f)
                        for f in os.listdir(EXAMPLES_DIR) if f.endswith(".csv"))
                 if os.path.isdir(EXAMPLES_DIR) else [])

DESCRIPTION = """
Upload a 12-lead ECG (100 Hz, 10 s, 1000×12 CSV) or try a preloaded PTB-XL
example. Predicts probabilities across 5 superclasses:

- **NORM** — Normal ECG
- **MI**   — Myocardial Infarction
- **STTC** — ST/T Change
- **CD**   — Conduction Disturbance
- **HYP**  — Hypertrophy

Per-class thresholds were tuned on the validation fold. Multi-label: more
than one class can be active. *Research/demo only — not a medical device.*
"""

with gr.Blocks(title="ECG Cardiovascular Disease Classifier") as demo:
    gr.Markdown("# ECG Cardiovascular Disease Classifier")
    gr.Markdown(DESCRIPTION)

    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(label="Upload ECG CSV", file_types=[".csv"])
            submit_btn = gr.Button("Classify", variant="primary")
            label_output = gr.Label(label="Probabilities", num_top_classes=5)
            diag_output = gr.Markdown()
        with gr.Column(scale=2):
            plot_output = gr.Plot(label="ECG Waveform (12 leads)")

    if example_files:
        gr.Examples(
            examples=[[f] for f in example_files],
            inputs=[file_input],
            outputs=[label_output, plot_output, diag_output],
            fn=predict_ecg, cache_examples=True,
            label="Try a preloaded PTB-XL example",
        )

    submit_btn.click(fn=predict_ecg, inputs=[file_input],
                     outputs=[label_output, plot_output, diag_output])

if __name__ == "__main__":
    demo.launch()
'''

with open(os.path.join(SPACE_DIR, "app.py"), "w") as f:
    f.write(app_py)
print(f"Wrote {os.path.join(SPACE_DIR, 'app.py')}")

Wrote hf_space/app.py


In [9]:
requirements = """keras>=3.13.0
tensorflow==2.20.0
numpy>=2.0.2
pandas>=2.2.3
matplotlib>=3.9.2
huggingface_hub>=0.33.5
"""
with open(os.path.join(SPACE_DIR, "requirements.txt"), "w") as f:
    f.write(requirements)

space_readme = f"""---
title: ECG Cardiovascular Disease Classifier
emoji: 🫀
colorFrom: red
colorTo: blue
sdk: gradio
sdk_version: 5.44.1
python_version: "3.12"
app_file: app.py
pinned: false
license: mit
tags:
  - ecg
  - cardiology
  - ptb-xl
  - keras
  - multi-label
---

# ECG Cardiovascular Disease Classifier

Live demo of a 1D-CNN trained on the [PTB-XL dataset](https://physionet.org/content/ptb-xl/)
for multi-label classification of 12-lead ECGs across 5 diagnostic superclasses:
**NORM, MI, STTC, CD, HYP**.

## How to use

1. Upload a CSV with shape **1000 rows × 12 columns** (100 Hz, 10 seconds,
   standard 12-lead order: I, II, III, aVR, aVL, aVF, V1–V6), **or**
2. Click one of the preloaded PTB-XL examples.

## Model

Model, normalisation parameters, and per-class decision thresholds are
pulled from [`{HF_MODEL_REPO}`](https://huggingface.co/{HF_MODEL_REPO}) at startup.

## Disclaimer

Research/coursework demo — **not a medical device, not for clinical use**.

*Authors: Edvard Vindenes Steenslid & Morten Kvamme — DAT255*
"""
with open(os.path.join(SPACE_DIR, "README.md"), "w") as f:
    f.write(space_readme)

print("Space staging dir contents:")
for root, dirs, files in os.walk(SPACE_DIR):
    for fn in sorted(files):
        p = os.path.join(root, fn)
        size = os.path.getsize(p) / 1024
        rel = os.path.relpath(p, SPACE_DIR)
        print(f"  {rel:35s} {size:>7.1f} KB")

Space staging dir contents:
  README.md                               1.1 KB
  app.py                                  5.2 KB
  requirements.txt                        0.1 KB
  examples/mi_example.csv                82.9 KB
  examples/norm_example.csv              83.3 KB
  examples/sttc_example.csv              82.8 KB


## Push Space files to HuggingFace

In [10]:
create_repo(HF_SPACE_REPO, repo_type="space", space_sdk="gradio", exist_ok=True)

# Files that may linger from older Space versions
STALE_SPACE_FILES = [
    "examples/norm_example.csv",
    "examples/mi_example.csv",
    "examples/sttc_example.csv",
    "examples/cd_example.csv",
    "examples/hyp_example.csv",
    "app_old.py",
]

try:
    existing_space = set(api.list_repo_files(repo_id=HF_SPACE_REPO, repo_type="space"))
except Exception as e:
    print(f"(Space is new or unreachable, skipping cleanup: {e})")
    existing_space = set()

# We re-delete example CSVs that we're about to re-upload.
# This ensures Gradio's example cache (gradio_cached_examples/) gets invalidated on the next build.
to_delete_space = [f for f in STALE_SPACE_FILES if f in existing_space]

# Also nuke any cached Gradio examples left over from earlier builds
cached_examples = [f for f in existing_space
                   if f.startswith("gradio_cached_examples/")]
to_delete_space.extend(cached_examples)

if to_delete_space:
    print(f"Deleting {len(to_delete_space)} stale file(s) from {HF_SPACE_REPO}:")
    for f in to_delete_space:
        print(f"  - {f}")
        api.delete_file(path_in_repo=f, repo_id=HF_SPACE_REPO,
                        repo_type="space",
                        commit_message=f"Remove stale {f}")
else:
    print("No stale files to delete in Space repo.")

# Push fresh Space files
upload_folder(
    folder_path=SPACE_DIR,
    repo_id=HF_SPACE_REPO,
    repo_type="space",
    commit_message=f"Deploy {DEPLOY_MODEL} with val-tuned thresholds + examples",
)
print(f"\nPushed to https://huggingface.co/spaces/{HF_SPACE_REPO}")

Deleting 3 stale file(s) from Steenslid/ecg-ptbxl-demo:
  - examples/norm_example.csv
  - examples/mi_example.csv
  - examples/sttc_example.csv

Pushed to https://huggingface.co/spaces/Steenslid/ecg-ptbxl-demo
The Space will rebuild automatically — first build takes ~5 minutes.


## Pretend we're a user

Fetch everything back from HF Hub and run one prediction end-to-end.

In [11]:
fresh_model = keras.saving.load_model(
    hf_hub_download(HF_MODEL_REPO, "ecg_model.keras"), compile=False)
fresh_params = np.load(hf_hub_download(HF_MODEL_REPO, "normalisation_params.npz"))
with open(hf_hub_download(HF_MODEL_REPO, "thresholds.json")) as f:
    fresh_thr = json.load(f)

# Read one of our own example CSVs
example_csv = os.path.join(EXAMPLES_DIR, "norm_example.csv")
ecg = pd.read_csv(example_csv).to_numpy(dtype=np.float32)
assert ecg.shape == (1000, 12), f"unexpected shape: {ecg.shape}"

ecg_norm = (ecg - fresh_params["mean"].squeeze()) / fresh_params["std"].squeeze()
probs = fresh_model.predict(ecg_norm[np.newaxis], verbose=0).squeeze()

print("Smoke test on norm_example.csv:")
for i, sc in enumerate(SUPERCLASSES):
    flag = "✓" if probs[i] >= fresh_thr[sc] else " "
    print(f"  [{flag}] {sc:5s} prob={probs[i]:.3f}  thr={fresh_thr[sc]:.2f}")
print("\nIf NORM is flagged, deployment works end-to-end.")

ecg_model.keras:   0%|          | 0.00/8.43M [00:00<?, ?B/s]

thresholds.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

W0000 00:00:1776522130.776541   47506 hlo_rematerialization.cc:3204] Can't reduce memory use below 9.11GiB (9786614078 bytes) by rematerialization; only reduced to 16.19GiB (17382631456 bytes), down from 16.19GiB (17382631456 bytes) originally
W0000 00:00:1776522140.836662   47058 bfc_allocator.cc:502] Allocator (GPU_0_bfc) ran out of memory trying to allocate 16.19GiB (rounded to 17382375936)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
I0000 00:00:1776522140.836760   47058 bfc_allocator.cc:1049] BFCAllocator dump for GPU_0_bfc
I0000 00:00:1776522140.836765   47058 bfc_allocator.cc:1056] Bin (256): 	Total Chunks: 38, Chunks in use: 37. 9.5KiB allocated for chunks. 9.2KiB in use in bin. 2.3KiB client-requested in use in bin.
I0000 00:00:1776522140.836769   47058 bfc_allocator.cc:1056] Bin (512): 	Total Chunks: 1

Smoke test on norm_example.csv:
  [✓] NORM  prob=0.733  thr=0.27
  [ ] MI    prob=0.102  thr=0.35
  [ ] STTC  prob=0.032  thr=0.45
  [ ] CD    prob=0.189  thr=0.44
  [ ] HYP   prob=0.009  thr=0.27

If NORM is flagged, deployment works end-to-end.


---

Space: `https://huggingface.co/spaces/{HF_SPACE_REPO}`.

### Rollback

If a future training run produces a worse model.
Revert via the Files tab on the model repo page, or `huggingface-cli`:

```bash
huggingface-cli repo revert <commit-hash> --repo-id {HF_MODEL_REPO} --repo-type model
```

The Space will pick up the reverted model on its next cold start.